# More complex aggregations

You can also perform complex aggregations on your data, involving grouping and joins, using the MongoDB Query API.

As you have seen, this is done using aggregation pipelines, which sequentially process documents using a series of data processing "stages".

Let's run some aggregation pipelines against the `movies` collection.

**Re-run the cells below to install `pymongo` and access the movies collection from the database.**

In [11]:
!pip install  --quiet pymongo==4.13.2


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [12]:
import os
from pymongo import MongoClient

MONGODB_URI = os.environ["MONGODB_URI"]

In [13]:
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI)

# Access the sample_mflix database
db = mongodb_client["sample_mflix"]

# Access the movies collection
collection = db["movies"]

### Top 5 movies with the highest IMDB ratings

To get the top 5 movies with the highest IMDB ratings, you need to:
1. Sort the documents in descending order of `imdb.rating`.
2. Limit the results to 5.
3. Project out only the `title` field in the final results.

Each of the steps above corresponds to a stage in the aggregation pipeline. As with CRUD operations, you will use different operators available in the MongoDB Query API to build the aggregation pipeline. For example, you will use `$sort` to sort the results in a particular order, `$limit` to limit the number of results returned, and `$project` to project out only certain fields.

![](images/agg_1.png)

Refer to MongoDB's official documentation to learn more about these operators:
* [`$sort`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/sort/)
* [`$limit`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/limit/)
* [`$project`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/project/)

**Create the aggregation pipeline to list the top 5 movies with highest IMDB ratings consisting of `$sort`, `$limit` and `$project` stages.**

In [14]:
sort_query = [
          { "$sort": { "imdb.rating": -1 } },
          { "$limit": 5 },
          { "$project": { "title": 1, "_id": 0} } ]

for doc in collection.aggregate(sort_query):
    print(doc)

{'title': 'The Danish Girl'}
{'title': 'Landet som icke èr'}
{'title': 'Scouts Guide to the Zombie Apocalypse'}
{'title': 'Catching the Sun'}
{'title': 'La nao capitana'}


### Top 5 directors by average IMDB rating, who have made at least 20 movies

Let's look at a slightly more complex aggregation pipeline. Here, we want to find the top 5 directors by average IMDB rating, who have made more than 20 movies. Once again, let's break down the calculation into sub-steps:
1. Extract individual directors from the array of `directors`.
2. For each director, count the number of films made, and the average `imdb.rating`.
3. Filter for directors who have made >= 20 movies.
4. Sort the resulting list of directors in descending ourder of average IMDB rating.
5. Limit the number of results returned to 5.

You learned about the `$sort` and `$limit` stages in the previous example. We will use these in Steps 4 and 5 of the pipeline, but you will need the following stages for Step 1 to 3:
1. [`$unwind`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/unwind/): Deconstruct a single document with a list of directors into multiple documents, one for each director.
2. [`$group`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/group/): Group documents by director, and calculate additional fields such as film count (`filmCount`) and average `imdb.rating` (`avgRating`)using accumulator expressions such as `$sum` and `$avg` respectively. Refer to MongoDB's documentation to see all the [accumulator expressions](https://www.mongodb.com/docs/manual/reference/operator/aggregation/group/#std-label-accumulators-group) supported by the MongoDB Query API.
3. [`$match`](https://www.mongodb.com/docs/manual/reference/operator/aggregation/match/): Filter directors based on condition, in this case, film count >= 20.

![](images/agg_2.png)

**Create the aggregation pipeline consisting of the `$unwind`, `$group`, `$match`, `$sort` and `$limit` stages.**

In [15]:
group_query = [ { "$unwind": "$directors" },
          { "$group": { "_id": "$directors", "filmCount": { "$sum": 1 }, "avgRating": { "$avg": "$imdb.rating" } } },
          { "$match": { "filmCount": { "$gte": 20 } } },
          { "$sort": { "avgRating": -1 } },
          { "$limit": 5 } ]

# Execute the aggregation pipeline and iterate through the resulting cursor
for doc in collection.aggregate(group_query):
    print(doc)

{'_id': 'William Wyler', 'filmCount': 21, 'avgRating': 7.676190476190476}
{'_id': 'Martin Scorsese', 'filmCount': 32, 'avgRating': 7.640625}
{'_id': 'Alfred Hitchcock', 'filmCount': 24, 'avgRating': 7.5874999999999995}
{'_id': 'Steven Spielberg', 'filmCount': 29, 'avgRating': 7.479310344827587}
{'_id': 'Woody Allen', 'filmCount': 40, 'avgRating': 7.215000000000001}
